# Web-FLARE Colab Runner

Run this notebook once per worker. Use a **different `WORKER_ID`** in each Colab session (0, 1, 2) and the same `NUM_WORKERS`.

- `all` = clone/pull branch + install + run assigned jobs + merge
- Outputs are written to `/content/drive/MyDrive/active-rag-webflare`
- Re-running the same cell resumes from `predictions.jsonl`
- The runner stops starting new examples when `BUDGET_MINUTES` is exhausted
- The committed `data/pilot/` split means no HuggingFace dataset download is needed

Requires a GPU runtime (Qwen3-4B is loaded in 4-bit).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

WORKER_ID = 0          # change to 1 and 2 in the other Colab sessions
NUM_WORKERS = 3
BUDGET_MINUTES = 75
REPO = '/content/active-rag'
EXTRA = []             # e.g. ['--quick'] to cap every job at 50 examples

if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run([
        'git', 'clone', '--branch', 'webflare', '--depth', '1',
        'https://github.com/PIYUSH-SATIYA/active-rag.git', REPO,
    ], check=True)

os.chdir(REPO)

subprocess.run([
    'python', 'scripts/colab_runner.py', 'all',
    '--worker-id', str(WORKER_ID),
    '--num-workers', str(NUM_WORKERS),
    '--budget-minutes', str(BUDGET_MINUTES),
    *EXTRA,
], check=True)

## Merge results

Run this cell in any session after the workers finish. It is safe to run with partial coverage; the summary reports `N` and `Coverage` columns.

Outputs:

- `/content/drive/MyDrive/active-rag-webflare/summary.md`
- `/content/drive/MyDrive/active-rag-webflare/summary.csv`
- `/content/drive/MyDrive/active-rag-webflare/summary.json`

In [ ]:
import os
import subprocess

REPO = '/content/active-rag'
os.chdir(REPO)

subprocess.run(['python', 'scripts/colab_runner.py', 'merge'], check=True)

from IPython.display import Markdown, display
summary_path = '/content/drive/MyDrive/active-rag-webflare/summary.md'
if os.path.exists(summary_path):
    display(Markdown(open(summary_path).read()))

## Notes

- Jobs are assigned longest-first to the least-loaded worker, so the default matrix splits 100/100/100 examples across three workers.
- Default jobs: 2Wiki web (100), StrategyQA web (100), 2Wiki web single-retrieval (50), 2Wiki web minus Wikipedia (50). BM25/no-retrieval baselines are re-scored from `data/pilot/baselines`.
- `--include-optional` adds the hybrid job (Pyserini + a multi-GB prebuilt index). Expect its setup to dominate the budget.
- `--quick` caps every job at 50 examples; use it if the first run reports low coverage.
- `EXTRA = ['--dry-run']` prints the job/ID assignment without loading the model.
- Separate Google accounts: after the run, pack the results and share the zip:

```python
!python scripts/colab_runner.py pack --pack-out /content/drive/MyDrive/webflare-worker0.zip
```

Then on the merging machine:

```python
!python scripts/colab_runner.py merge --merge-roots shared/worker1 shared/worker2
```